### Maximal Marginal Relevance
MMR (Maximal Marginal Relevance) is a powerful diversity-aware retrieval technique used in information retrieval and RAG pipelines to balance relevance and novelty when selecting documents.

In [2]:
import warnings
warnings.filterwarnings("ignore")

In [4]:
from langchain_community.vectorstores import FAISS
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain

## Step 1: Load and chunk the document

In [ ]:
loader = TextLoader("langchain_rag_dataset.txt")
raw_docs = loader.load()
splitter = RecursiveCharacterTextSplitter(
    chunk_size = 300, 
    chunk_overlap = 50
)

chunks = splitter.split_documents(raw_docs)
chunks

[Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain is an open-source framework designed to simplify the development of applications using large language models (LLMs).\nLangChain provides abstractions for working with prompts, chains, memory, and agents, making it easier to build complex LLM-based systems.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='The framework supports integration with various vector databases like FAISS and Chroma for semantic retrieval.\nLangChain enables Retrieval-Augmented Generation (RAG) by allowing developers to fetch relevant context before generating responses.'),
 Document(metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
 Document(metadata={'

## Step 2: FAISS Vector Store with HuggingFace Embeddings

In [8]:
embedding_model = OllamaEmbeddings(model="nomic-embed-text-v2-moe:latest")
vectorstore = FAISS.from_documents(chunks, embedding_model)

## Step 3: Create MMR Retirever

In [9]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3}
)

## Step 4: Prompt and LLM

In [ ]:
prompt = PromptTemplate.from_template("""
Answer the question based on the context provided.

Context:
{context}

Question: {input}
""")
llm = ChatOllama(model = "gemma3:latest")

## Step 5: RAG Pipeline

In [11]:
document_chain = create_stuff_documents_chain(llm = llm, prompt = prompt)
rag_chain = create_retrieval_chain(retriever = retriever, combine_docs_chain = document_chain)

## Step 6: Query

In [12]:
query = {"input": "How does LangChain support agents and memory?"}
response = rag_chain.invoke(query)

print("✅ Answer:\n", response["answer"])

✅ Answer:
 According to the context, LangChain supports agents by allowing them to use tools like calculators, search APIs, and custom functions based on instructions. It also supports memory through its models’ ability to retain previous interactions, making multi-turn conversations more coherent. RAG pipelines further enhance this by combining document retrieval with LLM response generation.


In [13]:
response

{'input': 'How does LangChain support agents and memory?',
 'context': [Document(id='36a7c697-6b96-4d65-b368-f0df8be9459d', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Memory in LangChain helps models retain previous interactions, making multi-turn conversations more coherent.\nAgents in LangChain can use tools like calculators, search APIs, or custom functions based on the instructions they receive.'),
  Document(id='a111a6ba-affa-4ec0-9561-4d495a357be9', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='Chroma is a lightweight vector store often used in LangChain for embedding-based document storage and retrieval.\nPrompt templates in LangChain support Jinja-style formatting and variable injection to customize model inputs.'),
  Document(id='03bf4529-2d68-4b5c-89bd-551f6b2df3db', metadata={'source': 'langchain_rag_dataset.txt'}, page_content='LangChain agents can interact with external APIs and databases, enhancing the capabilities of LLM-powered app